# AGENTS_003 - Policy Comparison with vLLM on AMD MI300X

Workshop-style notebook for **Policy & Document Comparison Assistant**.

This follows the AMD AI Academy pattern (vLLM + Pydantic AI), adapted for our hackathon agent:

| Academy workshop | This project |
|------------------|--------------|
| Conversational agent + MCP tools | **Fixed pipeline** (parse -> diff -> LLM analysis) |
| Airbnb / time MCP servers | **Native Python services** (no MCP needed) |
| Agent chooses tools at runtime | Orchestrator calls each step in order |

### Do we need MCP?

**No - not for the core demo.** MCP helps when an LLM must *dynamically* pick external tools. Our app uses a fixed workflow: upload PDFs -> structural diff -> semantic LLM -> regulatory impact.

---

## Table of contents
1. Launch vLLM server
2. Install dependencies
3. Pydantic AI smoke test
4. Native tool + policy comparison
5. Run full comparison pipeline

## Step 1: Launch vLLM server

Open a **terminal** and start vLLM:

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
    --served-model-name Qwen3-30B-A3B \
    --api-key abc-123 \
    --port 8000 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes \
    --trust-remote-code
```

Monitor GPU: `watch rocm-smi`

> **Port note:** vLLM uses **8000**. FastAPI backend uses **8080**.

In [1]:
import os
import sys
from pathlib import Path

def resolve_agent_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "app" / "main.py").is_file() and candidate.name == "document-comparison-agent":
            return candidate
        if (candidate / "agents" / "document-comparison-agent" / "app" / "main.py").is_file():
            return candidate / "agents" / "document-comparison-agent"
    raise RuntimeError(
        "Could not find document-comparison-agent. Start Jupyter from the agent or TheRock repo."
    )

AGENT_ROOT = resolve_agent_root()
THEROCK_ROOT = AGENT_ROOT.parent.parent if AGENT_ROOT.parent.name == "agents" else AGENT_ROOT.parent

if str(AGENT_ROOT) not in sys.path:
    sys.path.insert(0, str(AGENT_ROOT))

from app.env_loader import load_agent_env

load_agent_env()

BASE_URL = os.environ.get("BASE_URL", "http://localhost:8000/v1")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "abc-123")
LLM_MODEL = os.environ.get("LLM_MODEL", "Qwen3-30B-A3B")

os.environ["BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["LLM_MODEL"] = LLM_MODEL
os.environ["USE_LLM"] = os.environ.get("USE_LLM", "true")

print("Config set:")
print("  BASE_URL     =", BASE_URL)
print("  LLM_MODEL    =", LLM_MODEL)
print("  AGENT_ROOT   =", AGENT_ROOT)
print("  THEROCK_ROOT =", THEROCK_ROOT)

Config set:
  BASE_URL       = http://localhost:8000/v1
  LLM_MODEL      = Qwen3-30B-A3B
  PROJECT_ROOT   = C:\Users\Rajeswari\.gemini\antigravity\scratch\AMD-TCS-Hackthon
  BACKEND_ROOT   = C:\Users\Rajeswari\.gemini\antigravity\scratch\AMD-TCS-Hackthon\backend


In [2]:
import httpx

response = httpx.get(
    f"{BASE_URL}/models",
    headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
    timeout=30.0,
)
response.raise_for_status()
models = [m["id"] for m in response.json().get("data", [])]
print("Available models:", models)
assert LLM_MODEL in models, f"{LLM_MODEL} not found - check vLLM --served-model-name"
print(f"\nOK: {LLM_MODEL} is ready at {BASE_URL}")

ConnectError: [WinError 10061] No connection could be made because the target machine actively refused it

## Step 2: Install dependencies

In [ ]:
!pip install -q -r "{AGENT_ROOT / 'requirements-notebook.txt'}"

## Step 3: Pydantic AI smoke test (Academy pattern)

Verify Qwen3 responds through vLLM before running the comparison pipeline.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel(LLM_MODEL, provider=provider)

agent = Agent(
    model=agent_model,
    system_prompt="You are a regulatory policy analyst. Answer concisely.",
)

In [ ]:
result = await agent.run("What is the capital of France?")
print(result.output)

## Step 4: Native tool + policy comparison

Pydantic AI `@Tool` wrapping our comparison function (Academy Step 4 style, no MCP).

In [ ]:
import asyncio

from pydantic_ai import Tool
from app.services.pipeline import run_comparison

SAMPLES = AGENT_ROOT / "data" / "samples"
LEGACY = SAMPLES / "legacy_policy.pdf"
MODERN = SAMPLES / "modernized_policy.pdf"


@Tool
def compare_sample_policies() -> str:
    """Compare the built-in legacy vs modernized privacy policy PDFs and return executive summary."""
    if not LEGACY.is_file() or not MODERN.is_file():
        return (
            "Sample PDFs missing. Run: "
            "python agents/document-comparison-agent/scripts/make_sample_pdfs.py"
        )

    async def _run():
        return await run_comparison(
            LEGACY.name,
            LEGACY.read_bytes(),
            MODERN.name,
            MODERN.read_bytes(),
        )

    result = asyncio.run(_run())
    return (
        f"LLM used: {result.llm_used}\n"
        f"Stats: {result.stats}\n\n"
        f"Executive summary:\n{result.executive_summary}"
    )


policy_agent = Agent(
    model=agent_model,
    tools=[compare_sample_policies],
    system_prompt=(
        "You help compare regulatory policy documents.\n"
        "When asked to compare sample policies, call compare_sample_policies()."
    ),
)

print("Policy agent ready.")

In [ ]:
result = await policy_agent.run("Compare the sample legacy and modernized privacy policies.")
print(result.output)

## Step 5: Run the full comparison pipeline directly

Same pipeline used by the FastAPI backend - deterministic orchestration.

In [3]:
if not LEGACY.is_file():
    !python "{AGENT_ROOT / 'scripts' / 'make_sample_pdfs.py'}"

NameError: name 'LEGACY' is not defined

In [4]:
from importlib import reload
import app.config as config_module
import app.llm.client as llm_module

reload(config_module)
reload(llm_module)

from app.config import settings
from app.llm.client import LLMClient
from app.services.pipeline import run_comparison

print("App LLM config:")
print("  base_url:", settings.llm_base_url)
print("  model:   ", settings.llm_model)
print("  use_llm: ", settings.use_llm)

ok, msg = await LLMClient().ping()
status = "OK" if ok else "FAIL"
print(f"\nLLM ping: {status} - {msg}")

ModuleNotFoundError: No module named 'app'

In [5]:
comparison = await run_comparison(
    LEGACY.name,
    LEGACY.read_bytes(),
    MODERN.name,
    MODERN.read_bytes(),
)

print("LLM used:", comparison.llm_used)
print("Alignment score:", comparison.alignment_score)
if comparison.format_warnings:
    print("\nFormat warnings:")
    for w in comparison.format_warnings:
        print(f"  - {w}")
print("Stats:", comparison.stats)
print("\nExecutive summary:")
print(comparison.executive_summary)
print("\nTop semantic differences:")
for diff in comparison.semantic_differences[:3]:
    print(f"  - [{diff.severity.value}] {diff.section_title}: {diff.summary}")

NameError: name 'run_comparison' is not defined